# CNN ベースライン学習ノートブック

USDJPY 5分足方向予測 - CNN ベースラインモデル学習

**Kaggle commit mode 対応**: このノートブックは Save & Run All で実行完了します。

- 入力: `/kaggle/input/pearless-usdjpy-m5/` の numpy 配列
- 出力: `/kaggle/working/best_model.pt`（AC-013）
- ログ: `/kaggle/working/logs/training_log_cnn_{timestamp}.csv`（AC-022/023）
- 目的: PatchTST / iTransformer との比較基準（AC-021）

In [ ]:
# 依存パッケージのインストール（Kaggle 環境向け）
# Kaggle にはデフォルトで PyTorch がインストール済みのため、
# 追加パッケージのみインストールする
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "scikit-learn"], check=True
)

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch

# Kaggle 環境では /kaggle/input/{dataset-slug}/ にデータが配置される
DATASET_DIR = Path("/kaggle/input/pearless-usdjpy-m5")
WORKING_DIR = Path("/kaggle/working")

# ローカル環境でのフォールバック（テスト実行時）
if not DATASET_DIR.exists():
    DATASET_DIR = Path("data")

# ソースコードを sys.path に追加（Kaggle でコードをコピーした場合の対応）
REPO_ROOT = Path("/kaggle/working/pearless")
if not REPO_ROOT.exists():
    REPO_ROOT = Path("..")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Dataset dir: {DATASET_DIR}")
print(f"Working dir: {WORKING_DIR}")

In [ ]:
# データ読み込み（AC-013: Kaggle Dataset から numpy 配列をロード）
X_train: np.ndarray = np.load(DATASET_DIR / "X_train.npy")
y_train: np.ndarray = np.load(DATASET_DIR / "y_train.npy")
X_val: np.ndarray = np.load(DATASET_DIR / "X_val.npy")
y_val: np.ndarray = np.load(DATASET_DIR / "y_val.npy")

print(f"X_train shape: {X_train.shape}, dtype: {X_train.dtype}")
print(f"y_train shape: {y_train.shape}, dtype: {y_train.dtype}")
print(f"X_val shape:   {X_val.shape},   dtype: {X_val.dtype}")
print(f"y_val shape:   {y_val.shape},   dtype: {y_val.dtype}")

# shape 検証（AC-002）
assert X_train.ndim == 3 and X_train.shape[1] == 60 and X_train.shape[2] == 16, (
    f"X_train shape 不正: {X_train.shape}"
)
assert X_val.ndim == 3 and X_val.shape[1] == 60 and X_val.shape[2] == 16, (
    f"X_val shape 不正: {X_val.shape}"
)

# クラス分布確認
unique, counts = np.unique(y_train, return_counts=True)
print("\nクラス分布 (y_train):")
for cls, cnt in zip(unique, counts):
    label = ["UP", "DOWN", "NEUTRAL"][cls]
    print(f"  {label}({cls}): {cnt} ({cnt / len(y_train) * 100:.1f}%)")

In [ ]:
# CNN ベースラインモデルの初期化
from models.cnn import CNNModel

model = CNNModel(
    seq_len=60,
    n_features=16,
    n_classes=3,
)

# パラメータ数確認（≤ 10M）
total_params = sum(p.numel() for p in model.parameters())
assert total_params <= 10_000_000, f"パラメータ数が 10M 超: {total_params:,}"
print(f"パラメータ数: {total_params:,} (≤ 10M OK)")

# forward shape 動作確認（AC-021）
model.eval()
with torch.no_grad():
    x_test = torch.randn(4, 60, 16)
    output = model(x_test)
assert output.shape == (4, 3), f"shape 不正: {output.shape}"
assert torch.allclose(output.sum(dim=1), torch.ones(4), atol=1e-4), (
    "softmax 合計が 1 でない"
)
assert output.min() >= 0.0, "確率が負値"
print(f"AC-021 OK: output.shape={output.shape}, softmax sum≈1.0")

In [ ]:
# 学習実行（AC-013: /kaggle/working/ にチェックポイント保存）
from models.training import train

train(
    model=model,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    model_name="cnn",
    n_epochs=100,
    batch_size=256,
    lr=1e-4,
    weight_decay=1e-4,
    patience=15,
    checkpoint_dir=str(WORKING_DIR),
    log_dir=str(WORKING_DIR / "logs"),
    scheduler_t0=10,
)

In [ ]:
# チェックポイント保存確認（AC-013/014）
best_model_path = WORKING_DIR / "best_model.pt"
assert best_model_path.exists(), f"best_model.pt が存在しない: {best_model_path}"
print(f"AC-013 OK: チェックポイント保存確認: {best_model_path}")

# 保存したモデルをロードして動作確認
model_loaded = CNNModel(
    seq_len=60,
    n_features=16,
    n_classes=3,
)
model_loaded.load_state_dict(torch.load(best_model_path, map_location="cpu"))
model_loaded.eval()

with torch.no_grad():
    x_verify = torch.randn(2, 60, 16)
    output_verify = model_loaded(x_verify)

assert output_verify.shape == (2, 3), f"ロード後 shape 不正: {output_verify.shape}"
print(f"AC-014 OK: モデルロード後の推論確認 shape={output_verify.shape}")

# ログファイル確認（AC-022）
log_dir = WORKING_DIR / "logs"
log_files = list(log_dir.glob("training_log_cnn_*.csv"))
assert len(log_files) > 0, f"ログファイルが見つからない: {log_dir}"
print(f"AC-022 OK: ログファイル確認: {log_files[0]}")

print("\n全 AC 確認完了。Kaggle commit mode 実行成功。")